# Exercise 01 — Pandas in Python

![Pandas illustration](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTkKmkBqviJKR3yCj5F251eRodlrKmubG6ey7pJMGGLMs2CF23gBT_4QevLGRVUjcSXSkQ&usqp=CAU)

**Pandas** is a Python library for working with structured data. It helps us load, inspect, filter, analyse and export tables. Like a spreadsheet, it organises information into rows and columns; unlike manually editing cells in Excel, we can describe a sequence of operations in Python and repeat it whenever the data changes.

Pandas is particularly useful when working with datasets containing many records. We will use a prepared dataset of Flickr photographs to explore the basics. Later, we can use similar techniques with data acquired through an API.

You can find further examples in the [Pandas documentation](https://pandas.pydata.org/docs/).

## Understanding a DataFrame

The main Pandas object is a **DataFrame**: a table of rows and columns. Each **row** represents a record (for example, a photograph); each **column** represents an attribute (such as the title, date or location). A single column is called a **Series**. The **index** identifies rows, starting at 0 by default.

![Diagram of a Pandas DataFrame with rows, columns, a Series and a single value](assets/dataframe-explained.svg)

The diagram uses illustrative data. Our Flickr dataset contains different columns, which we will inspect shortly.

![Original DataFrame illustration](assets/Dataframe.png)

## Reading a CSV file

A **CSV** (comma-separated values) file stores a table as plain text. We will load `Flickr.csv`, which is included in the `assets` folder beside this notebook.

First, import Pandas. The name `pd` is a common short alias: we use `pd.read_csv()` instead of writing `pandas.read_csv()` each time.

In [ ]:
import pandas as pd
from pathlib import Path

### Where is the file?

File paths look different on different operating systems. Windows commonly uses a backslash (`assets\\Flickr.csv`), while macOS and Linux use a forward slash (`assets/Flickr.csv`). Python's built-in `pathlib` library lets us write one expression that works on either system:

```python
path = Path("assets") / "Flickr.csv"
```

Here, `Path("assets")` refers to a folder, and the `/` **joins the folder name and file name**. It is not division. Python uses the appropriate separators for your operating system.

This is a **relative path**: Python looks for `assets` inside its current working directory. For the example to work, run the notebook with its Day 2 Pandas folder as the working directory. In a cloud notebook such as Colab, the dataset must first be uploaded or made available in that runtime.

In [ ]:
path = Path("assets") / "Flickr.csv"

print(path)
print(Path.cwd())  # Folder Python is currently using

### Load and inspect the table

`pd.read_csv()` reads the file and returns a DataFrame. We assign it to a variable called `df`. The method `head()` displays the first five rows, so we can see what the data looks like without printing the entire dataset.

In [ ]:
df = pd.read_csv(path)

df.head()

Two quick ways to explore a new dataset are `shape`, which reports the number of rows and columns, and `columns`, which lists the column names.

In [ ]:
print(df.shape)
print(df.columns)

## Accessing data

Let's begin by selecting rows and columns from our Flickr DataFrame. The following methods are useful when exploring any new dataset.

### Previewing rows

`head()` shows the first rows; `tail()` shows the last. We can specify how many rows to display.

In [ ]:
df.head(4)

In [ ]:
df.tail(4)

### Selecting a column

Use square brackets with a column name to retrieve a **Series**, such as the photograph titles. You can then select one value from that Series.

In [ ]:
df["title"].head()

In [ ]:
df["title"].iloc[0]  # Title from the first row

## Selecting rows: `.loc` and `.iloc`

**Every DataFrame has an index.** By default, Pandas creates an index with the labels `0, 1, 2, ...`. But an index does not have to be numbered consecutively: we can choose another column to provide the labels.

- **`.iloc` means integer location:** select by **row position**. The first row is always at position `0`, regardless of its index label.
- **`.loc` means label location:** select by **index label**, which may be a number, a string, a date or another value.

When a DataFrame has the default index, `df.iloc[0]` and `df.loc[0]` usually select the same row. This makes the difference easy to miss.

In [ ]:
df.iloc[0]  # First row by position

In [ ]:
df.loc[0]   # Row whose index label is 0

### Use the Flickr photo ID as the index

The Flickr CSV includes a column called `p_id`, containing each photograph's ID. In our original DataFrame, `p_id` is an ordinary **column**; the index is the automatically generated row labels.

We can use `set_index("p_id")` to create a second DataFrame whose **index labels are photo IDs**. We will call it `df_by_id`, keeping the original `df` unchanged.

In [ ]:
df_by_id = df.set_index("p_id")

df_by_id.head()

Now the row **positions** are still `0, 1, 2, ...`, but the **index labels** are Flickr photo IDs. We can compare the first record in two ways. First, get its ID, then use that ID as the label. Both expressions below refer to the same photograph, but they identify it differently.

In [ ]:
first_photo_id = df_by_id.index[0]

print("First row position:", 0)
print("First photo ID (index label):", first_photo_id)

In [ ]:
df_by_id.iloc[0]  # First row by position

In [ ]:
df_by_id.loc[first_photo_id]  # Row with this photo ID as its index label

Notice that `df_by_id.loc[0]` would look for a photograph whose **ID is exactly 0**; it would not mean the first row. Unless such a label exists, that expression raises a `KeyError`.

Unlike `.iloc`, `.loc` does **not** require index labels to be unique. If several rows share a label, `.loc` can return all of them. Unique photo IDs are convenient here because they normally identify individual photographs.

We can also locate photographs **without changing the index**, by comparing values in a column. For example:

In [ ]:
df.loc[df["p_id"] == first_photo_id]

This last expression is called **Boolean filtering**: the comparison creates a True/False result for every row, and `.loc` keeps the rows where the condition is true. We will use filtering again later in the notebook.

## Descriptive statistics

Once we have loaded a dataset, we often want a quick overview of its numerical values. Pandas provides statistical functions that work directly on DataFrame columns.

### Summarising a numerical column

The method `.describe()` calculates several statistics at once. Here we use the `lat` column, which stores the latitude of each Flickr photograph.

In [ ]:
df["lat"].describe()

The result includes:

| Statistic | Meaning |
| --- | --- |
| `count` | Number of non-missing values |
| `mean` | Arithmetic average |
| `std` | Standard deviation, describing how widely values vary |
| `min` | Smallest value |
| `25%` | First quartile: 25% of values are at or below this value |
| `50%` | Median: the middle value |
| `75%` | Third quartile: 75% of values are at or below this value |
| `max` | Largest value |

The `count` may be smaller than the total number of rows because missing values are not included.

Latitude is a convenient numerical example, but an average latitude is not necessarily a meaningful description of where photographs are concentrated. Always consider what a statistic tells us about the data.

### Individual statistics

We can ask for a particular statistic without calculating the full summary:

In [ ]:
print("Minimum latitude:", df["lat"].min())
print("Maximum latitude:", df["lat"].max())
print("Mean latitude:", df["lat"].mean())

### Counting repeated values

For a text or categorical column, `.value_counts()` counts how frequently each value appears. Here we look at photograph titles; `.head(10)` displays the ten most frequently repeated titles. Missing values are excluded by default.

In [ ]:
df["title"].value_counts().head(10)

If the dataset includes an `owner` column, the same method can reveal which photographers have contributed the most images:

```python
if "owner" in df.columns:
    print(df["owner"].value_counts().head(10))
```

This is a useful distinction: numerical statistics describe the **range and distribution of quantities**, while frequency counts help us understand repeated categories or labels.

[Further reference: Pandas descriptive statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics).

## Data types and conversion

Pandas assigns a **data type** to each column when it reads a CSV file. A column might contain whole numbers (`int64`), decimal numbers (`float64`), text (`object` or `string`), or dates (`datetime64`). These types determine which operations are possible. For example, we can calculate the average of numerical values, but a date stored as ordinary text cannot be used reliably for date comparisons.

### Inspect the data types

The `.dtypes` attribute displays the data type of each column in our Flickr DataFrame.

In [ ]:
df.dtypes

### Convert a column

Sometimes a number is really an identifier, rather than something we want to calculate with. Flickr photo IDs (`p_id`), for example, are identifiers. We can represent them as strings using `.astype()`.

Here we create a separate Series, leaving the original DataFrame unchanged so we can continue using its photo IDs in later examples.

In [ ]:
photo_ids_as_text = df["p_id"].astype("string")

print(photo_ids_as_text.head())
print(photo_ids_as_text.dtype)

### Working with dates

A date in a CSV file may initially be read as text. For example, `2013-10-05 16:27:16` looks like a date to us, but Python needs to recognise it as a **datetime** value before we can reliably sort or filter by time.

The function `pd.to_datetime()` converts the values in `p_date` to dates. Some datasets contain missing or invalid entries. Instead of stopping with an error, `errors="coerce"` turns entries that cannot be converted into `NaT` (*Not a Time*), Pandas' marker for missing dates.

In [ ]:
df["p_date"] = pd.to_datetime(df["p_date"], errors="coerce")

df["p_date"].head()

We should inspect how many dates could not be converted. The expression `.isna()` checks which values are missing; `.sum()` counts them.

In [ ]:
missing_dates = df["p_date"].isna().sum()
print("Missing or invalid dates:", missing_dates)
print("Total records:", len(df))

We have **not deleted any rows**. The original records remain in the DataFrame, with `NaT` wherever the date is unavailable or invalid. We can now use the valid dates to filter photographs by time in the next section.

If we later need a dataset containing only records with valid dates, we can explicitly create a separate version with `df.dropna(subset=["p_date"])`. This is preferable to deleting an unexplained row by its position.

[Further reference: Pandas time series and date handling](https://pandas.pydata.org/docs/user_guide/timeseries.html).

## Filtering, sorting and exporting data

We can now apply what we have learned to a small data-processing workflow: select Flickr photographs from **March 2004**, arrange them chronologically and save the results as a new CSV file. The original dataset remains unchanged.

### 1. Filter photographs by date

The `.query()` method selects rows that satisfy a condition. We use two comparisons to specify the beginning and end of March 2004:

In [ ]:
filtered_df = df.query(
    "p_date >= '2004-03-01' and p_date < '2004-04-01'"
)

filtered_df.head()

The first comparison includes photographs from **1 March onwards** (`>=`). The second excludes photographs taken on or after **1 April** (`<`). This way we include every time of day on 31 March without having to specify the last second of the month.

Because we converted `p_date` to a datetime type in the previous section, Pandas can compare the dates correctly. Missing dates (`NaT`) are not included in the result.

### 2. Sort the results

The method `.sort_values()` arranges the rows by a chosen column. Here `by="p_date"` specifies the date column, and `ascending=True` puts earlier photographs first. Use `ascending=False` to reverse the order.

In [ ]:
filtered_df = filtered_df.sort_values(
    by="p_date",
    ascending=True
)

filtered_df.head()

We can use `len()` to count the records that matched the filter:

In [ ]:
print(f"Photographs found: {len(filtered_df)}")

### 3. Export the selection as CSV

The method `.to_csv()` saves our filtered DataFrame as a new file. We use the `Path` approach introduced earlier to place the result in the `assets` folder.

By default, Pandas also writes the DataFrame index to a CSV file. Setting `index=False` avoids adding an extra index column. The original `Flickr.csv` remains untouched.

In [ ]:
output_path = Path("assets") / "Filtered.csv"

filtered_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

We have now completed a simple data workflow: **load → inspect → convert → filter → sort → export**. The exported CSV can be opened in a spreadsheet, loaded again into Pandas, or used in another application such as QGIS.

[Further reference: Pandas `to_csv()` documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).